In [1]:
import pandas as pd
import numpy as np
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib.pyplot as plt

In [10]:
data_log2 = pd.read_pickle('data_log2_Coimbra_threshold.pkl')
import pickle

with open('list_groups_Coimbra.pkl', 'rb') as f:
    list_groups = pickle.load(f)

print(len(list_groups))


list_groups = pd.Series(list_groups)

55


In [3]:
import os
print("Path lettura:", os.path.abspath('data_log2_Coimbra_threshold.pkl'))

Path lettura: /home/jovyan/Project_lisboa/work/data_log2_Coimbra_threshold.pkl


In [4]:
len(list_groups)

55

In [5]:
data_log2

Protein.Group,A0A075B6H7,A0A075B6K5,A0A075B6P5;P01615;A0A087WW87;P01614,A0A075B6S5;A0A0C4DH67;A0A0C4DH69,A0A0A0MRZ8;P04433,A0A0A0MS15,A0A0B4J1U7,A0A0B4J1X8,A0A0B4J1Y9,A0A0B4J2B5,...,Q9NYQ8,Q9P121,Q9P2S2,Q9UBP4,Q9UBX5,Q9UKI9;P09086;P14859,Q9ULB1,Q9Y4C0,Q9Y646,Q9Y6R7
109094,10.707256,5.411257,10.023893,7.638458,9.985671,5.338670,7.244240,9.311739,9.095131,10.772999,...,4.290432,8.415112,6.759808,11.329084,6.049101,9.030046,6.976387,7.458193,6.139852,6.967906
103208,10.510289,4.121894,9.891121,6.968736,9.974615,6.571228,6.063140,8.986320,8.741339,11.256226,...,4.580694,8.203358,6.722548,11.352380,6.357804,10.300444,6.761179,7.385923,6.398954,7.078834
106086,10.989870,5.323903,10.223121,7.522158,10.246075,6.168520,6.770049,9.322539,9.478424,11.403204,...,4.759955,8.712379,6.741656,11.591747,6.424029,9.908852,7.325251,7.835930,6.582637,7.505629
105634,11.706315,6.707000,11.341958,8.700062,11.423557,7.464734,5.915024,9.812152,9.418742,12.369186,...,4.100952,8.368140,6.500428,11.666797,6.222213,10.495495,6.355367,7.287186,6.226406,6.764420
106008,10.733066,4.704031,10.645001,8.005720,10.433419,5.200661,8.202614,9.718603,9.274483,11.513303,...,4.073829,8.467292,6.067114,11.768011,6.594716,10.377319,7.064355,7.343239,6.230420,7.659339
110203,10.245838,4.292936,9.241104,7.304219,10.118993,6.080417,8.769074,9.116656,8.588032,10.729825,...,5.029488,8.565658,6.868131,11.799646,6.323964,9.477698,7.183040,7.686171,6.173755,7.017710
108382,11.131838,5.668079,10.281976,7.698288,9.901271,6.452478,9.130254,9.069530,9.252261,11.197180,...,3.923415,8.317399,6.656554,11.519843,6.304646,10.041673,6.712032,7.335998,6.424015,7.993810
104968,10.706228,5.402255,10.087489,7.788953,9.729106,6.248379,8.236249,9.583534,8.581683,11.398770,...,4.286970,8.294180,6.557016,11.373409,5.980238,9.407812,6.957938,7.421526,6.570718,8.275324
107018,10.330312,5.685836,10.257482,7.405022,10.142733,6.457701,8.621645,9.630560,9.432364,11.696972,...,3.918405,8.469768,6.735170,11.248604,6.273708,10.364966,6.913189,7.510329,5.884200,6.478033
109456,9.491047,2.964088,8.829406,6.604319,9.283452,4.231041,4.319119,8.628099,7.791254,10.291482,...,4.311648,8.384715,6.565285,11.583666,6.443028,9.050121,7.160134,7.625658,6.226703,6.323552


In [6]:
def calculate_vip(pls, X):
    t = pls.x_scores_
    w = pls.x_weights_
    q = pls.y_loadings_

    p, h = w.shape
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    total_s = np.sum(s)

    vip = np.zeros((p,))

    for i in range(p):
        weight = np.array([
            (w[i, j] ** 2) * s[j] for j in range(h)
        ])
        vip[i] = np.sqrt(p * (np.sum(weight) / total_s))

    return vip

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, matthews_corrcoef
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import LabelEncoder

# ======================
# DATA LOADING
# ======================
X = data_log2
y = np.array(list_groups)

# ======================
# GLOBAL PARAMETERS
# ======================
vip_thresholds = [0.1, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
decision_thresholds = [0.4, 0.45, 0.5, 0.55, 0.6]

n_iterations = 50  
n_seeds = 15      

all_test_mcc = []
all_test_auc = []
all_n_features = []
all_best_vip_thr = []
all_best_dec_thr = []
global_stable_counts = pd.Series(0, index=X.columns)

def stratified_bootstrap(X, y):
    ad_idx = np.where(y == "MCI-AD")[0]
    ct_idx = np.where(y == "MCI-CT")[0]
    ad_sample = np.random.choice(ad_idx, size=len(ad_idx), replace=True)
    ct_sample = np.random.choice(ct_idx, size=len(ct_idx), replace=True)
    indices = np.concatenate([ad_sample, ct_sample])
    np.random.shuffle(indices)
    return X.iloc[indices], y[indices]

# ======================
# MAIN SEED LOOP
# ======================
for seed in range(n_seeds):
    print(f"\n" + "="*40)
    print(f"RUNNING SEED {seed}")
    print("="*40)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=seed
    )

    cv = StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=seed)
    le = LabelEncoder()
    le.fit(y_train)
    y_train_enc = le.transform(y_train)
    y_test_enc = le.transform(y_test)

    # Griglia per ottimizzazione incrociata (VIP x Decision)
    grid_mcc = np.zeros((len(vip_thresholds), len(decision_thresholds)))
    grid_auc = np.zeros(len(vip_thresholds))

    for v_idx, v_thr in enumerate(vip_thresholds):
        fold_mccs = []
        fold_aucs = []

        for train_idx, val_idx in cv.split(X_train, y_train):
            X_sub, y_sub = X_train.iloc[train_idx], y_train[train_idx]
            X_val, y_val = X_train.iloc[val_idx], y_train[val_idx]
            y_val_enc = le.transform(y_val)

            # --- Feature Selection (VIP Stability) ---
            vip_matrix = np.zeros((n_iterations, X_sub.shape[1]))
            for i in range(n_iterations):
                X_b, y_b = stratified_bootstrap(X_sub, y_sub)
                pls_b = PLSRegression(n_components=2)
                pls_b.fit(X_b, le.transform(y_b))
                vip_matrix[i] = (calculate_vip(pls_b, X_b) > 1).astype(int)

            vip_freq = vip_matrix.mean(axis=0)
            sel_feats = X_sub.columns[vip_freq >= v_thr]
            if len(sel_feats) == 0: sel_feats = [X_sub.columns[np.argmax(vip_freq)]]

            # --- Validation ---
            pls_cv = PLSRegression(n_components=2)
            pls_cv.fit(X_sub[sel_feats], le.transform(y_sub))
            y_prob_val = pls_cv.predict(X_val[sel_feats]).ravel()

            mccs_per_dec = [matthews_corrcoef(y_val_enc, (y_prob_val > d_thr).astype(int)) for d_thr in decision_thresholds]
            fold_mccs.append(mccs_per_dec)
            fold_aucs.append(roc_auc_score(y_val_enc, y_prob_val))

        grid_mcc[v_idx, :] = np.mean(fold_mccs, axis=0)
        grid_auc[v_idx] = np.mean(fold_aucs)

    # Identificazione best combination
    best_v_idx, best_d_idx = np.unravel_index(np.argmax(grid_mcc), grid_mcc.shape)
    best_v_thr = vip_thresholds[best_v_idx]
    best_d_thr = decision_thresholds[best_d_idx]

    # --- Training Finale (Full X_train) ---
    final_vip_matrix = np.zeros((n_iterations, X_train.shape[1]))
    for i in range(n_iterations):
        X_b, y_b = stratified_bootstrap(X_train, y_train)
        pls_b = PLSRegression(n_components=2).fit(X_b, le.transform(y_b))
        final_vip_matrix[i] = (calculate_vip(pls_b, X_b) > 1).astype(int)
    
    final_vip_freq = final_vip_matrix.mean(axis=0)
    top_features = X_train.columns[final_vip_freq >= best_v_thr]
    if len(top_features) == 0: top_features = [X_train.columns[np.argmax(final_vip_freq)]]

    global_stable_counts.loc[top_features] += 1

    # --- Test Finale (X_test) ---
    pls_final = PLSRegression(n_components=2).fit(X_train[top_features], y_train_enc)
    y_prob_test = pls_final.predict(X_test[top_features]).ravel()
    
    test_mcc = matthews_corrcoef(y_test_enc, (y_prob_test > best_d_thr).astype(int))
    test_auc = roc_auc_score(y_test_enc, y_prob_test)

    all_test_mcc.append(test_mcc)
    all_test_auc.append(test_auc)
    all_n_features.append(len(top_features))
    all_best_vip_thr.append(best_v_thr)
    all_best_dec_thr.append(best_d_thr)

    print(f"Seed {seed}: MCC={test_mcc:.3f}, AUC={test_auc:.3f}, Features={len(top_features)}")

# ======================
# FINAL PERFORMANCE SUMMARY
# ======================
print("\n" + "="*50)
print("FINAL CONSOLIDATED RESULTS (30 SEEDS)")
print("="*50)
print(f"Mean MCC: {np.mean(all_test_mcc):.4f} (±{np.std(all_test_mcc):.4f})")
print(f"Mean AUC: {np.mean(all_test_auc):.4f} (±{np.std(all_test_auc):.4f})")
print(f"Mean Features: {np.mean(all_n_features):.2f} (±{np.std(all_n_features):.2f})")

print("\n--- TOP 30 MOST STABLE PROTEINS ACROSS SEEDS ---")
global_stability = (global_stable_counts / n_seeds).sort_values(ascending=False)
print(global_stability)

# ======================
# BUILD RESULT TABLE (PLS-DA)
# ======================
df_plsda = pd.DataFrame({
    "model": "PLS-DA",
    "seed": list(range(n_seeds)),
    "mcc": all_test_mcc,
    "auc": all_test_auc,
    "n_features": all_n_features,
    "best_vip_threshold": all_best_vip_thr,
    "best_decision_threshold": all_best_dec_thr
})

print("\n--- FREQUENCY OF SELECTED VIP THRESHOLDS (Stability) ---")
print(pd.Series(all_best_vip_thr).value_counts(normalize=True).sort_index())

print("\n--- FREQUENCY OF SELECTED DECISION THRESHOLDS (Classification) ---")
print(pd.Series(all_best_dec_thr).value_counts(normalize=True).sort_index())


RUNNING SEED 0
Seed 0: MCC=1.000, AUC=1.000, Features=167

RUNNING SEED 1
Seed 1: MCC=1.000, AUC=1.000, Features=157

RUNNING SEED 2
Seed 2: MCC=1.000, AUC=1.000, Features=163

RUNNING SEED 3
Seed 3: MCC=1.000, AUC=1.000, Features=159

RUNNING SEED 4
Seed 4: MCC=1.000, AUC=1.000, Features=152

RUNNING SEED 5
Seed 5: MCC=1.000, AUC=1.000, Features=164

RUNNING SEED 6
Seed 6: MCC=1.000, AUC=1.000, Features=154

RUNNING SEED 7
Seed 7: MCC=1.000, AUC=1.000, Features=160

RUNNING SEED 8
Seed 8: MCC=1.000, AUC=1.000, Features=157

RUNNING SEED 9
Seed 9: MCC=0.874, AUC=1.000, Features=154

RUNNING SEED 10
Seed 10: MCC=1.000, AUC=1.000, Features=159

RUNNING SEED 11
Seed 11: MCC=1.000, AUC=1.000, Features=164

RUNNING SEED 12
Seed 12: MCC=1.000, AUC=1.000, Features=155

RUNNING SEED 13
Seed 13: MCC=1.000, AUC=1.000, Features=160

RUNNING SEED 14
Seed 14: MCC=1.000, AUC=1.000, Features=155

FINAL CONSOLIDATED RESULTS (30 SEEDS)
Mean MCC: 0.9916 (±0.0314)
Mean AUC: 1.0000 (±0.0000)
Mean Feature

In [8]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, matthews_corrcoef
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import LabelEncoder

# ======================
# DATA
# ======================
X = data_log2
y = np.array(list_groups)

# ======================
# PARAMETERS
# ======================
vip_thresholds = [0.1, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
decision_thresholds = [0.4, 0.45, 0.5, 0.55, 0.6]

n_iterations = 50
n_splits = 5

# ======================
# ENCODING
# ======================
le = LabelEncoder()
y_enc = le.fit_transform(y)

# ======================
# STRATIFIED BOOTSTRAP
# ======================
def stratified_bootstrap(X, y):
    ad_idx = np.where(y == "MCI-AD")[0]
    ct_idx = np.where(y == "MCI-CT")[0]
    ad_sample = np.random.choice(ad_idx, size=len(ad_idx), replace=True)
    ct_sample = np.random.choice(ct_idx, size=len(ct_idx), replace=True)
    indices = np.concatenate([ad_sample, ct_sample])
    np.random.shuffle(indices)
    return X.iloc[indices], y[indices]

# ======================
# CV
# ======================
cv = StratifiedShuffleSplit(n_splits=n_splits, test_size=0.3, random_state=42)

grid_mcc = np.zeros((len(vip_thresholds), len(decision_thresholds)))
grid_auc = np.zeros(len(vip_thresholds))

# 👉 per salvare metriche fold-level della best combo
all_val_mcc = []
all_val_auc = []

# 👉 conteggio globale VIP>1 (VERO, non threshold-based)
vip_global_counts = np.zeros(X.shape[1])
vip_global_total = 0

for v_idx, v_thr in enumerate(vip_thresholds):
    fold_mccs = []
    fold_aucs = []

    for train_idx, val_idx in cv.split(X, y):
        X_sub, y_sub = X.iloc[train_idx], y[train_idx]
        X_val, y_val = X.iloc[val_idx], y[val_idx]
        y_val_enc = le.transform(y_val)

        # ======================
        # VIP STABILITY
        # ======================
        vip_matrix = np.zeros((n_iterations, X_sub.shape[1]))

        for i in range(n_iterations):
            X_b, y_b = stratified_bootstrap(X_sub, y_sub)

            pls_b = PLSRegression(n_components=2)
            pls_b.fit(X_b, le.transform(y_b))

            vip_scores = calculate_vip(pls_b, X_b)

            # 👉 CONTEGGIO GLOBALE (indipendente da threshold)
            vip_global_counts += (vip_scores > 1).astype(int)
            vip_global_total += 1

            vip_matrix[i] = (vip_scores > 1).astype(int)

        vip_freq = vip_matrix.mean(axis=0)
        sel_feats = X_sub.columns[vip_freq >= v_thr]

        if len(sel_feats) == 0:
            sel_feats = [X_sub.columns[np.argmax(vip_freq)]]

        # ======================
        # VALIDATION
        # ======================
        pls_cv = PLSRegression(n_components=2)
        pls_cv.fit(X_sub[sel_feats], le.transform(y_sub))

        y_prob_val = pls_cv.predict(X_val[sel_feats]).ravel()

        mccs_per_dec = [
            matthews_corrcoef(y_val_enc, (y_prob_val > d_thr).astype(int))
            for d_thr in decision_thresholds
        ]

        fold_mccs.append(mccs_per_dec)
        fold_aucs.append(roc_auc_score(y_val_enc, y_prob_val))

    grid_mcc[v_idx, :] = np.mean(fold_mccs, axis=0)
    grid_auc[v_idx] = np.mean(fold_aucs)

# ======================
# BEST PARAMS
# ======================
best_v_idx, best_d_idx = np.unravel_index(np.argmax(grid_mcc), grid_mcc.shape)
best_v_thr = vip_thresholds[best_v_idx]
best_d_thr = decision_thresholds[best_d_idx]

print(f"\nBest VIP threshold: {best_v_thr}")
print(f"Best decision threshold: {best_d_thr}")

# ======================
# RICALCOLO METRICHE SOLO PER BEST COMBO
# ======================
for train_idx, val_idx in cv.split(X, y):
    X_sub, y_sub = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    y_val_enc = le.transform(y_val)

    vip_matrix = np.zeros((n_iterations, X_sub.shape[1]))

    for i in range(n_iterations):
        X_b, y_b = stratified_bootstrap(X_sub, y_sub)
        pls_b = PLSRegression(n_components=2)
        pls_b.fit(X_b, le.transform(y_b))
        vip_matrix[i] = (calculate_vip(pls_b, X_b) > 1).astype(int)

    vip_freq = vip_matrix.mean(axis=0)
    sel_feats = X_sub.columns[vip_freq >= best_v_thr]

    if len(sel_feats) == 0:
        sel_feats = [X_sub.columns[np.argmax(vip_freq)]]

    pls_cv = PLSRegression(n_components=2)
    pls_cv.fit(X_sub[sel_feats], le.transform(y_sub))

    y_prob_val = pls_cv.predict(X_val[sel_feats]).ravel()

    all_val_mcc.append(
        matthews_corrcoef(y_val_enc, (y_prob_val > best_d_thr).astype(int))
    )
    all_val_auc.append(
        roc_auc_score(y_val_enc, y_prob_val)
    )

# ======================
# METRICHE FINALI
# ======================
print("\n=== VALIDATION PERFORMANCE ===")
print(f"MCC: {np.mean(all_val_mcc):.4f} ± {np.std(all_val_mcc):.4f}")
print(f"AUC: {np.mean(all_val_auc):.4f} ± {np.std(all_val_auc):.4f}")

# ======================
# VIP GLOBAL FREQUENCY
# ======================
vip_global_freq = vip_global_counts / vip_global_total
vip_global_series = pd.Series(vip_global_freq, index=X.columns).sort_values(ascending=False)

print("\n=== VIP > 1 GLOBAL FREQUENCY ===")
print(vip_global_series)


Best VIP threshold: 0.1
Best decision threshold: 0.4

=== VALIDATION PERFORMANCE ===
MCC: 1.0000 ± 0.0000
AUC: 1.0000 ± 0.0000

=== VIP > 1 GLOBAL FREQUENCY ===
Protein.Group
P02766               1.000000
P02749               1.000000
P61916               1.000000
P02751               1.000000
P18065               1.000000
                       ...   
Q9P2S2               0.016889
P01857               0.016000
P04211;A0A075B6I9    0.011111
A0A075B6H7           0.010222
P10451               0.009778
Length: 237, dtype: float64


In [9]:
vip_global_series.to_pickle('vip_global_series_Coimbra_threshold.pkl')